# Agente de IA Minimalista - v3.2: análise temporal (Gemini)

**O que esta versão acrescenta ao notebook do professor:**

1. **Nova tool `analisar_serie_temporal`**: recebe um serviço e uma métrica e devolve *quando* a métrica mudou de
   comportamento (ponto de mudança), a taxa de crescimento depois disso e quais eventos/deploys aconteceram
   perto desse momento (antes ou depois).
2. **Novo problema (cenário B, INC-2026-0912):** vazamento de memória gradual no `catalog-api`, cujo sintoma só aparece
   horas depois no `checkout-api`, com um deploy recente e inocente no checkout como "pista falsa".
3. **Séries temporais no mockup** (geradas de forma determinística, sempre os mesmos números).
4. **Testes e avaliação:** o agente roda várias vezes em cada cenário, com e sem a tool temporal, e a tabela final mostra
   acerto, rodadas, tokens e custo.
5. **Portado para Gemini** (API nativa, sem bibliotecas extras) e com nova tentativa automática em erros 429/503.
6. **Instruções melhoradas** a partir de falhas observadas no teste do cenário original (veja o relatório).
7. **Robustez (v2.1), motivada por falhas reais dos testes:** mais tentativas e esperas maiores quando a API está sobrecarregada (erro 503); "guarda de qualidade" que pede o relatório completo quando o modelo responde curto demais; tratamento de resposta vazia; avaliação que separa erro de infraestrutura de erro do agente e registra o *motivo* de cada falha; timeouts de rede tratados como falha transitória; resultados salvos após cada execução, com retomada (v2.2).
8. **Melhorias da v3, cada uma motivada por um dado da 1ª avaliação (9 execuções):** (a) *cenário C*, mais difícil, porque o B era resolvido sem a tool temporal em 2 de 3 tentativas; (b) última rodada reservada ao relatório e aviso de orçamento, porque 1 execução estourou as 8 rodadas sem relatório; (c) chamadas repetidas não são reexecutadas (economia de tokens); (d) limite de saída de 1200 para 2000 tokens, porque 1 relatório foi cortado (`MAX_TOKENS`); (e) regras novas: pedir rollback quando houver causa (3 de 4 falhas terminaram sem pedir), e nunca declarar 100% de confiança; (f) a avaliação agora mede também se o relatório *menciona* o deploy correto (métrica fraca: só detecta o ID no texto, não quem ele culpa), e guarda o texto de cada relatório.
10. **v3.2, motivada por dois erros vistos nos relatórios do cenário C:** o agente analisava só o serviço do alarme e, quando via a origem certa, ainda culpava o deploy mais recente. A varredura agora checa se cada deploy posterior **alterou a inclinação** da métrica de origem, e as instruções ganharam uma regra de decisão.
9. **v3.1, motivada pelo diagnóstico do cenário C (0/3 acertos):** a tool temporal ganhou *varredura* (ordem em que as métricas mudaram) e *aviso de métrica que mudou antes*; as instruções passaram a proibir atribuir a causa ao deploy mais próximo do sintoma.

In [1]:
import sys
print("Kernel OK. Python:", sys.version.split()[0])

Kernel OK. Python: 3.11.9


In [2]:
import json
import math
import os
import random
import time
import urllib.error
import urllib.request
from copy import deepcopy
from getpass import getpass
from pathlib import Path

ARQUIVO_CHAVE = Path("gemini_key.txt")   # arquivo de texto com a chave, na mesma pasta do notebook

if not os.environ.get("GEMINI_API_KEY"):
    if ARQUIVO_CHAVE.exists():
        os.environ["GEMINI_API_KEY"] = ARQUIVO_CHAVE.read_text(encoding="utf-8").strip()
        print("Chave lida do arquivo gemini_key.txt")
    else:
        os.environ["GEMINI_API_KEY"] = getpass("GEMINI_API_KEY: ").strip()

if not os.environ["GEMINI_API_KEY"]:
    raise RuntimeError("Chave vazia. Coloque a chave no arquivo gemini_key.txt.")
print(f"Chave carregada ({len(os.environ['GEMINI_API_KEY'])} caracteres).")

BASE_URL = "https://generativelanguage.googleapis.com/v1beta"
MODEL = "gemini-3.1-flash-lite"
MAX_RODADAS = 8
MAX_OUTPUT_TOKENS = 2000
THINKING_LEVEL = "low"

# Plano gratuito: custo real zero. Precos de referencia (GPT-5.6 Luna, do notebook original)
# usados so para calcular o "custo equivalente".
LIMITE_ESTIMADO_USD = 0.02
PRECO_INPUT_1M = 0.20
PRECO_OUTPUT_1M = 1.20

Chave lida do arquivo gemini_key.txt
Chave carregada (53 caracteres).


## Mockup: utilitários de tempo e gerador de séries temporais

Cada série é gerada com `random.Random(semente)`; portanto **os números são sempre os mesmos** em qualquer computador.

In [3]:
SERVICOS = ["api-gateway", "recommendation-api", "catalog-api", "checkout-api"]
METRICAS = ["cpu_pct", "memoria_pct", "latencia_p95_ms", "taxa_erro_pct"]
SD_RUIDO = {"cpu_pct": 1.2, "memoria_pct": 0.5, "taxa_erro_pct": 0.05}   # latencia: 3% do valor


def hhmm_para_min(h):
    hh, mm = h.split(":")
    return int(hh) * 60 + int(mm)


def min_para_hhmm(m):
    m = int(round(m))
    return f"{m // 60:02d}:{m % 60:02d}"


def rampa(t, t0, duracao, delta):
    return delta * min(1.0, max(0.0, (t - t0) / duracao))


def gerar_series(inicio, fim, passo, funcao_valor, semente):
    rng = random.Random(semente)
    tempos = range(hhmm_para_min(inicio), hhmm_para_min(fim) + 1, passo)
    dados = {s: {m: [] for m in METRICAS} for s in SERVICOS}
    for t in tempos:
        for s in SERVICOS:
            for m in METRICAS:
                base = funcao_valor(s, m, t)
                sd = 0.03 * base if m == "latencia_p95_ms" else SD_RUIDO[m]
                dados[s][m].append(round(max(0.0, base + rng.gauss(0, sd)), 2))
    return {"inicio": inicio, "fim": fim, "passo_min": passo, "dados": dados}

## Cenário A: o original do professor (incidente INC-2026-0911)

Mantive os dados originais e apenas acrescentei uma série temporal coerente com eles (para a nova tool funcionar aqui também).

In [4]:
METRICAS_ATUAIS_A = {
    "api-gateway": {"cpu_pct": 38, "memoria_pct": 61, "latencia_p95_ms": 6200, "taxa_erro_pct": 18.4, "instancias": 3},
    "recommendation-api": {"cpu_pct": 98, "memoria_pct": 84, "latencia_p95_ms": 5870, "taxa_erro_pct": 31.7, "instancias": 48},
    "catalog-api": {"cpu_pct": 42, "memoria_pct": 55, "latencia_p95_ms": 93, "taxa_erro_pct": 0.2, "instancias": 4},
    "checkout-api": {"cpu_pct": 47, "memoria_pct": 58, "latencia_p95_ms": 310, "taxa_erro_pct": 0.6, "instancias": 5},
}


def valor_A(s, m, t):
    h = hhmm_para_min
    atual = METRICAS_ATUAIS_A[s][m]
    if s == "recommendation-api":
        antes = {"cpu_pct": 35, "memoria_pct": 52, "latencia_p95_ms": 180, "taxa_erro_pct": 0.4}[m]
        if m in ("cpu_pct", "memoria_pct"):
            return antes + rampa(t, h("13:58"), 5, atual - antes)
        return antes + rampa(t, h("14:00"), 3, atual - antes)
    if s == "api-gateway" and m in ("latencia_p95_ms", "taxa_erro_pct"):
        antes = 140 if m == "latencia_p95_ms" else 0.3
        return antes + rampa(t, h("14:03"), 2, atual - antes)
    return atual


SERIES_A = gerar_series("13:00", "14:08", 1, valor_A, semente=911)
for s in SERVICOS:                       # o ultimo ponto da serie e o valor atual do snapshot
    for m in METRICAS:
        SERIES_A["dados"][s][m][-1] = METRICAS_ATUAIS_A[s][m]

CENARIO_A = {
    "id": "A_original",
    "incidente": (
        "INC-2026-0911: desde 14h03, usuarios relatam lentidao no e-commerce. "
        "O api-gateway disparou alerta de latencia e o custo cloud esta subindo. "
        "Descubra a causa raiz e recomende a resposta operacional mais segura."
    ),
    "agora": "14:08",
    "metricas": METRICAS_ATUAIS_A,
    "logs": {
        "api-gateway": [
            "14:03:11 WARN upstream recommendation-api timeout after 5000ms",
            "14:03:14 WARN upstream recommendation-api timeout after 5000ms",
            "14:03:18 ERROR circuit breaker recommendation-api OPEN",
        ],
        "recommendation-api": [
            "14:02:57 WARN catalog response missing field related_items",
            "14:02:57 INFO invoking fallback target=recommendation-api depth=1",
            "14:02:58 INFO invoking fallback target=recommendation-api depth=8",
            "14:02:59 ERROR maximum fallback depth reached depth=25",
            "14:03:00 WARN request queue saturation=96%",
        ],
        "catalog-api": [
            "14:02:55 INFO GET /items/842 200 71ms",
            "14:03:01 INFO GET /items/163 200 66ms",
        ],
        "checkout-api": [
            "14:03:02 INFO POST /checkout 200 288ms",
            "14:03:07 INFO POST /checkout 200 302ms",
        ],
    },
    "deploys": {
        "api-gateway": [{"id": "dep-410", "versao": "5.8.1", "hora": "ontem 16:20", "status": "concluido"}],
        "recommendation-api": [
            {"id": "dep-442", "versao": "2.4.0", "hora": "13:58", "status": "concluido",
             "mudancas": ["novo fallback para resposta incompleta do catalogo",
                          "FALLBACK_SERVICE: catalog-api -> recommendation-api"]},
            {"id": "dep-431", "versao": "2.3.7", "hora": "ontem 11:40", "status": "concluido"},
        ],
        "catalog-api": [{"id": "dep-405", "versao": "3.1.2", "hora": "3 dias atras", "status": "concluido"}],
        "checkout-api": [{"id": "dep-398", "versao": "7.2.0", "hora": "5 dias atras", "status": "concluido"}],
    },
    "configuracoes": {
        "api-gateway": {"timeout_ms": 5000, "retries": 1},
        "recommendation-api": {"fallback_service": "recommendation-api", "fallback_max_depth": 25,
                               "autoscaling_min": 2, "autoscaling_max": 50, "autoscaling_cpu_target_pct": 60},
        "catalog-api": {"cache_ttl_s": 300, "replicas": 4},
        "checkout-api": {"replicas": 5, "timeout_ms": 1500},
    },
    "custos": {
        "api-gateway": {"baseline_usd_h": 0.22, "atual_usd_h": 0.24},
        "recommendation-api": {"baseline_usd_h": 0.31, "atual_usd_h": 7.44},
        "catalog-api": {"baseline_usd_h": 0.38, "atual_usd_h": 0.40},
        "checkout-api": {"baseline_usd_h": 0.51, "atual_usd_h": 0.52},
    },
    "linha_do_tempo": [
        {"hora": "13:58", "tipo": "deploy", "servico": "recommendation-api", "id": "dep-442"},
        {"hora": "14:01", "tipo": "autoscaling", "servico": "recommendation-api", "detalhe": "2 -> 18 instancias"},
        {"hora": "14:03", "tipo": "autoscaling", "servico": "recommendation-api", "detalhe": "18 -> 48 instancias"},
        {"hora": "14:03", "tipo": "alerta", "servico": "api-gateway", "detalhe": "latencia p95 acima de 5s"},
    ],
    "series": SERIES_A,
}
print("Cenario A pronto:", len(SERIES_A["dados"]["api-gateway"]["cpu_pct"]), "pontos por serie (1 min).")

Cenario A pronto: 69 pontos por serie (1 min).


## Cenário B (NOVO): vazamento de memória gradual (incidente INC-2026-0912)

**História (a que o agente precisa reconstruir):**
- 12:10: deploy `dep-451` no `catalog-api` troca o TTL do cache de 300 s para 0 (o cache nunca expira). A memória do catalog começa a crescer devagar.
- ~15:30: o heap chega a ~85% e começam pausas longas de coleta de lixo (GC); a latência do catalog dispara.
- ~15:39: o `checkout-api`, que depende do catalog, começa a ficar lento.
- 15:52: deploy `dep-455` no `checkout-api` (só um ajuste visual de banner). **É a pista falsa: ele aparece DEPOIS do início do sintoma.**
- 16:05: autoscaling do checkout (5 -> 8). 16:10: alerta de latência.

Sem olhar o tempo, o "último deploy" (dep-455) parece o culpado. A tool temporal mostra que o checkout já estava piorando antes dele
e que a memória do catalog mudou de tendência exatamente às 12:10.

In [5]:
def valor_B(s, m, t):
    h = hhmm_para_min
    t_deploy, t_gc = h("12:10"), h("15:30")
    mem_cat = 55 + (0.15 * (t - t_deploy) if t > t_deploy else 0)
    gc = max(0, t - t_gc)
    lat_cat = 90 + 12 * gc
    lat_chk = 310 + 1.8 * max(0, lat_cat - 200)
    if s == "catalog-api":
        return {"cpu_pct": 42 + 0.4 * gc, "memoria_pct": mem_cat,
                "latencia_p95_ms": lat_cat, "taxa_erro_pct": 0.2 + 0.05 * gc}[m]
    if s == "checkout-api":
        return {"cpu_pct": 47, "memoria_pct": 58, "latencia_p95_ms": lat_chk,
                "taxa_erro_pct": 0.6 + 0.12 * max(0, gc - 9)}[m]
    if s == "api-gateway":
        return {"cpu_pct": 38, "memoria_pct": 61,
                "latencia_p95_ms": 120 + 0.3 * max(0, lat_chk - 310), "taxa_erro_pct": 0.3}[m]
    return {"cpu_pct": 44, "memoria_pct": 57, "latencia_p95_ms": 150, "taxa_erro_pct": 0.4}[m]


SERIES_B = gerar_series("11:00", "16:15", 5, valor_B, semente=912)
INSTANCIAS_B = {"api-gateway": 3, "recommendation-api": 2, "catalog-api": 4, "checkout-api": 8}
METRICAS_ATUAIS_B = {
    s: {**{m: SERIES_B["dados"][s][m][-1] for m in METRICAS}, "instancias": INSTANCIAS_B[s]} for s in SERVICOS
}

CENARIO_B = {
    "id": "B_vazamento_de_memoria",
    "incidente": (
        "INC-2026-0912: desde cerca de 16h10, o checkout esta lento e o alerta de latencia p95 do checkout-api disparou. "
        "Clientes relatam demora para finalizar pedidos. "
        "Descubra a causa raiz e recomende a resposta operacional mais segura."
    ),
    "agora": "16:15",
    "metricas": METRICAS_ATUAIS_B,
    "logs": {
        "api-gateway": [
            "16:04 WARN upstream checkout-api slow response 1.4s",
            "16:11 WARN upstream checkout-api slow response 1.9s",
        ],
        "recommendation-api": ["16:10 INFO GET /recommendations 200 148ms"],
        "catalog-api": [
            "15:33 WARN GC pause 380ms heap=86%",
            "15:41 WARN GC pause 640ms heap=87%",
            "16:07 ERROR GC pause 1900ms heap=91%",
        ],
        "checkout-api": [
            "15:44 WARN call catalog-api /items latency 480ms",
            "16:03 WARN timeout calling catalog-api after 800ms retry=1",
            "16:09 ERROR POST /checkout 504 after 2100ms",
        ],
    },
    "deploys": {
        "api-gateway": [{"id": "dep-410", "versao": "5.8.1", "hora": "ontem 16:20", "status": "concluido"}],
        "recommendation-api": [{"id": "dep-431", "versao": "2.3.7", "hora": "ontem 11:40", "status": "concluido"}],
        "catalog-api": [
            {"id": "dep-451", "versao": "3.2.0", "hora": "12:10", "status": "concluido",
             "mudancas": ["cache de itens em memoria para acelerar listagens", "cache_ttl_s: 300 -> 0 (sem expiracao)"]},
            {"id": "dep-405", "versao": "3.1.2", "hora": "3 dias atras", "status": "concluido"},
        ],
        "checkout-api": [
            {"id": "dep-455", "versao": "7.3.0", "hora": "15:52", "status": "concluido",
             "mudancas": ["ajuste de texto e cor do banner promocional"]},
            {"id": "dep-398", "versao": "7.2.0", "hora": "5 dias atras", "status": "concluido"},
        ],
    },
    "configuracoes": {
        "api-gateway": {"timeout_ms": 5000, "retries": 1},
        "recommendation-api": {"autoscaling_min": 2, "autoscaling_max": 50},
        "catalog-api": {"cache_ttl_s": 0, "cache_max_entries": "ilimitado", "heap_max_mb": 4096, "replicas": 4},
        "checkout-api": {"replicas": 8, "timeout_ms": 1500, "catalog_timeout_ms": 800},
    },
    "custos": {
        "api-gateway": {"baseline_usd_h": 0.22, "atual_usd_h": 0.22},
        "recommendation-api": {"baseline_usd_h": 0.31, "atual_usd_h": 0.31},
        "catalog-api": {"baseline_usd_h": 0.38, "atual_usd_h": 0.38},
        "checkout-api": {"baseline_usd_h": 0.51, "atual_usd_h": 0.82},
    },
    "linha_do_tempo": [
        {"hora": "12:10", "tipo": "deploy", "servico": "catalog-api", "id": "dep-451"},
        {"hora": "15:52", "tipo": "deploy", "servico": "checkout-api", "id": "dep-455"},
        {"hora": "16:05", "tipo": "autoscaling", "servico": "checkout-api", "detalhe": "5 -> 8 instancias"},
        {"hora": "16:10", "tipo": "alerta", "servico": "checkout-api", "detalhe": "latencia p95 acima de 900ms"},
    ],
    "series": SERIES_B,
}

CENARIOS = {"A": CENARIO_A, "B": CENARIO_B}
ambiente = deepcopy(CENARIO_B)


def carregar_cenario(nome):
    global ambiente
    ambiente = deepcopy(CENARIOS[nome])
    return ambiente["id"]


print("Cenario B pronto:", len(SERIES_B["dados"]["catalog-api"]["memoria_pct"]), "pontos por serie (5 min).")
print("Snapshot atual do checkout-api:", METRICAS_ATUAIS_B["checkout-api"])

Cenario B pronto: 64 pontos por serie (5 min).
Snapshot atual do checkout-api: {'cpu_pct': 47.98, 'memoria_pct': 58.88, 'latencia_p95_ms': 1068.99, 'taxa_erro_pct': 4.86, 'instancias': 8}


## Cenário C (NOVO, v3): deploys confusos (incidente INC-2026-0913)

**Por que ele existe:** a primeira avaliação mostrou que o cenário B era fácil demais. Sem a tool temporal, o agente acertou 2 de 3 vezes,
porque a descrição do deploy `dep-451` ("cache sem expiração") entregava a resposta. No cenário C **tirei essas pistas textuais**.

**As séries de métricas são as mesmas do B.** O que muda são os deploys do `catalog-api`, que agora são dois e ambos parecem plausíveis:
- `dep-451` (12:10): "refatoração do módulo de listagem" (a causa real; a memória começa a crescer aqui);
- `dep-453` (15:25): "nível de log de INFO para DEBUG, sem reinicio" (pista falsa: a latência do catalog sobe 5 min depois dele);
- `dep-455` (15:52, no checkout): banner (pista falsa 2, vem depois do sintoma).

Só olhando a linha do tempo da **memória** dá para ver que ela já crescia 3 horas antes do `dep-453`.

In [6]:
CENARIO_C = deepcopy(CENARIO_B)
CENARIO_C["id"] = "C_deploys_confusos"
CENARIO_C["incidente"] = CENARIO_B["incidente"].replace("INC-2026-0912", "INC-2026-0913")
CENARIO_C["deploys"]["catalog-api"] = [
    {"id": "dep-453", "versao": "3.2.1", "hora": "15:25", "status": "concluido",
     "mudancas": ["ajuste do nivel de log do endpoint /items de INFO para DEBUG (a quente, sem reinicio)"]},
    {"id": "dep-451", "versao": "3.2.0", "hora": "12:10", "status": "concluido",
     "mudancas": ["refatoracao do modulo de listagem de itens", "atualizacao de dependencias internas"]},
    {"id": "dep-405", "versao": "3.1.2", "hora": "3 dias atras", "status": "concluido"},
]
CENARIO_C["configuracoes"]["catalog-api"] = {"heap_max_mb": 4096, "replicas": 4, "log_level_items": "DEBUG"}
CENARIO_C["linha_do_tempo"] = [
    {"hora": "12:10", "tipo": "deploy", "servico": "catalog-api", "id": "dep-451"},
    {"hora": "15:25", "tipo": "deploy", "servico": "catalog-api", "id": "dep-453"},
    {"hora": "15:52", "tipo": "deploy", "servico": "checkout-api", "id": "dep-455"},
    {"hora": "16:05", "tipo": "autoscaling", "servico": "checkout-api", "detalhe": "5 -> 8 instancias"},
    {"hora": "16:10", "tipo": "alerta", "servico": "checkout-api", "detalhe": "latencia p95 acima de 900ms"},
]
CENARIOS["C"] = CENARIO_C
print("Cenario C pronto (mesmas series do B; deploys do catalog-api sem pistas textuais).")

Cenario C pronto (mesmas series do B; deploys do catalog-api sem pistas textuais).


## Ferramentas

As 7 ferramentas originais continuam iguais, exceto `consultar_eventos_recentes`, que agora só mostra os eventos dos **últimos 90 minutos**
(como um painel real faria). Isso esconde o deploy das 12:10 e deixa em destaque o das 15:52.
A ferramenta nova é `analisar_serie_temporal`.

In [7]:
def _validar_servico(servico):
    if servico not in ambiente["metricas"]:
        return {"erro": f"Servico desconhecido: {servico}", "disponiveis": list(ambiente["metricas"])}
    return None


def consultar_metricas(servico):
    erro = _validar_servico(servico)
    return erro or {"servico": servico, **ambiente["metricas"][servico]}


def buscar_logs(servico):
    erro = _validar_servico(servico)
    return erro or {"servico": servico, "linhas": ambiente["logs"].get(servico, [])}


def consultar_deploys(servico):
    erro = _validar_servico(servico)
    return erro or {"servico": servico, "deploys": ambiente["deploys"].get(servico, [])}


def consultar_configuracao(servico):
    erro = _validar_servico(servico)
    return erro or {"servico": servico, "configuracao": ambiente["configuracoes"].get(servico, {})}


def consultar_custos(servico):
    erro = _validar_servico(servico)
    if erro:
        return erro
    custo = ambiente["custos"][servico]
    return {"servico": servico, **custo, "multiplicador": round(custo["atual_usd_h"] / custo["baseline_usd_h"], 1)}


def consultar_eventos_recentes():
    agora = hhmm_para_min(ambiente["agora"])
    recentes = [e for e in ambiente["linha_do_tempo"] if hhmm_para_min(e["hora"]) >= agora - 90]
    return {"agora": ambiente["agora"], "janela_min": 90, "eventos": recentes}


def solicitar_rollback(deploy_id, justificativa):
    # Esta ferramenta NAO executa o rollback. Ela abre uma solicitacao.
    encontrado = None
    for servico, deploys in ambiente["deploys"].items():
        for deploy in deploys:
            if deploy["id"] == deploy_id:
                encontrado = {"servico": servico, **deploy}
                break
    if not encontrado:
        return {"status": "rejeitado", "motivo": "deploy inexistente"}
    return {"status": "AGUARDANDO_APROVACAO_HUMANA", "deploy": encontrado,
            "justificativa_registrada": justificativa, "executado": False}

### A ferramenta nova: `analisar_serie_temporal`

**Atualização v3.1:** o teste do cenário C mostrou que o agente analisava só a *latência* (o sintoma), cuja mudança coincide com um deploy inocente. Por isso a tool ganhou a **varredura** (sem informar a métrica, mostra a *ordem* em que as métricas do serviço mudaram) e, quando se analisa uma só métrica, **avisa se outra métrica do mesmo serviço mudou antes**.

**Como funciona (em português simples):**
1. Pega a série da métrica (ex.: memória do catalog-api de 11:00 a 16:15).
2. Testa cada horário possível como "o momento em que a métrica mudou": antes dele a série é uma linha plana, depois dele é uma
   reta inclinada. O horário em que esse modelo "encaixa" melhor nos dados é o **ponto de mudança**.
3. Só declara que houve mudança se o encaixe for bom e a variação for grande em comparação com o ruído normal.
4. Lista os eventos (deploys, autoscaling, alertas) que aconteceram até 30 min antes ou depois desse ponto,
   dizendo se cada um veio **antes** ou **depois** da mudança. Um evento que veio depois não pode ser a causa.

In [8]:
def _ajustar_mudanca(tempos, valores):
    # Modelo: plano ate o instante tc; depois, reta partindo do nivel plano (continua em tc).
    n = len(valores)
    melhor = None
    for k in range(4, n - 3):
        a = sum(valores[:k]) / k
        tc = tempos[k]
        den = sum((tempos[i] - tc) ** 2 for i in range(k, n))
        b = sum((tempos[i] - tc) * (valores[i] - a) for i in range(k, n)) / den
        sse = sum((valores[i] - a) ** 2 for i in range(k))
        sse += sum((valores[i] - a - b * (tempos[i] - tc)) ** 2 for i in range(k, n))
        if melhor is None or sse < melhor[0]:
            melhor = (sse, k, a, b)
    return melhor


def _avaliar_metrica(servico, metrica):
    serie = ambiente["series"]
    valores = serie["dados"][servico][metrica]
    passo = serie["passo_min"]
    n = len(valores)
    tempos = [hhmm_para_min(serie["inicio"]) + i * passo for i in range(n)]
    sse, k, a, b = _ajustar_mudanca(tempos, valores)
    media = sum(valores) / n
    sse_plano = sum((v - media) ** 2 for v in valores)
    melhoria = 1 - sse / sse_plano if sse_plano > 0 else 0
    desvio_base = math.sqrt(sum((valores[i] - a) ** 2 for i in range(k)) / k)
    variacao_total = b * (tempos[-1] - tempos[k])
    houve = melhoria >= 0.5 and abs(variacao_total) > 4 * max(desvio_base, 1e-9)
    return {"valores": valores, "tempos": tempos, "passo": passo, "k": k, "a": a, "b": b, "media": media, "houve": houve}


def _eventos_proximos(tc, passo):
    tolerancia = max(passo, 3)   # a estimativa do ponto de mudanca tem margem de erro de alguns minutos
    proximos = []
    for evento in ambiente["linha_do_tempo"]:
        diferenca = hhmm_para_min(evento["hora"]) - tc
        if abs(diferenca) <= 30:
            posicao = ("no mesmo instante (dentro da margem de erro)" if abs(diferenca) <= tolerancia
                       else ("antes da mudanca" if diferenca < 0 else "depois da mudanca"))
            proximos.append({**evento, "diferenca_min": diferenca, "posicao": posicao})
    proximos.sort(key=lambda e: abs(e["diferenca_min"]))
    return proximos


def _inclinacao_janela(tempos, valores, t_ini, t_fim):
    pontos = [(t, v) for t, v in zip(tempos, valores) if t_ini <= t <= t_fim]
    if len(pontos) < 4:
        return None
    tm = sum(t for t, _ in pontos) / len(pontos)
    vm = sum(v for _, v in pontos) / len(pontos)
    den = sum((t - tm) ** 2 for t, _ in pontos)
    return sum((t - tm) * (v - vm) for t, v in pontos) / den * 60   # unidades por hora


def _resumo_mudanca(metrica, av):
    tc = av["tempos"][av["k"]]
    return {
        "metrica": metrica,
        "mudanca_de_tendencia": min_para_hhmm(tc),
        "baseline_antes_da_mudanca": round(av["a"], 2),
        "valor_atual": av["valores"][-1],
        "variacao_pct_vs_baseline": round((av["valores"][-1] - av["a"]) / av["a"] * 100, 1) if av["a"] else None,
        "inclinacao_apos_mudanca_por_hora": round(av["b"] * 60, 2),
    }


def analisar_serie_temporal(servico, metrica=None):
    erro = _validar_servico(servico)
    if erro:
        return erro
    if metrica is not None and metrica not in METRICAS:
        return {"erro": f"Metrica desconhecida: {metrica}", "disponiveis": METRICAS}

    serie = ambiente["series"]
    av = {m: _avaliar_metrica(servico, m) for m in METRICAS}
    passo = serie["passo_min"]
    tolerancia = max(passo, 3)
    base = {"servico": servico, "janela": f"{serie['inicio']}-{serie['fim']}", "resolucao_min": passo}

    # ---- modo varredura: nenhuma metrica informada -> ordem em que as metricas do servico mudaram ----
    if metrica is None:
        mudaram = sorted((a["tempos"][a["k"]], m) for m, a in av.items() if a["houve"])
        resultado = {**base, "modo": "varredura de todas as metricas do servico",
                     "ordem_de_mudanca": [_resumo_mudanca(m, av[m]) for _, m in mudaram],
                     "metricas_estaveis": [m for m, a in av.items() if not a["houve"]]}
        if mudaram:
            tc, primeira = mudaram[0]
            resultado["primeira_metrica_a_mudar"] = primeira
            resultado["eventos_proximos_da_primeira_mudanca"] = _eventos_proximos(tc, passo)
            a1 = av[primeira]
            posteriores = []
            for ev in ambiente["linha_do_tempo"]:   # deploys DESTE servico que vieram depois da primeira mudanca
                if ev["tipo"] != "deploy" or ev.get("servico") != servico:
                    continue
                te = hhmm_para_min(ev["hora"])
                if te <= tc + tolerancia:
                    continue
                b_antes = _inclinacao_janela(a1["tempos"], a1["valores"], max(te - 45, tc), te - 1)
                b_depois = _inclinacao_janela(a1["tempos"], a1["valores"], te, te + 45)
                item = {"id": ev.get("id"), "hora": ev["hora"]}
                if b_antes is None or b_depois is None:
                    item["alterou_a_inclinacao"] = "indeterminado (poucos pontos)"
                else:
                    limiar = 0.5 * abs(a1["b"] * 60)
                    item[f"inclinacao_{primeira}_antes_por_hora"] = round(b_antes, 2)
                    item[f"inclinacao_{primeira}_depois_por_hora"] = round(b_depois, 2)
                    item["alterou_a_inclinacao"] = abs(b_depois - b_antes) > limiar
                posteriores.append(item)
            if posteriores:
                resultado["deploys_posteriores_a_primeira_mudanca"] = posteriores
            resultado["leitura"] = ("A metrica que muda primeiro costuma ser a origem do problema; as que mudam depois "
                                    "podem ser consequencia. Um deploy posterior so pode ter agravado a primeira metrica "
                                    "se alterou_a_inclinacao for true. Confirme com deploys e logs.")
        return resultado

    # ---- modo metrica unica ----
    a = av[metrica]
    valores, tempos, k = a["valores"], a["tempos"], a["k"]
    amostras_passo = max(1, len(valores) // 12)
    indices = list(range(0, len(valores), amostras_passo))
    if indices[-1] != len(valores) - 1:
        indices.append(len(valores) - 1)
    amostras = [{"hora": min_para_hhmm(tempos[i]), "valor": valores[i]} for i in indices]

    resultado = {**base, "metrica": metrica, "valor_atual": valores[-1]}
    if not a["houve"]:
        resultado["tendencia"] = "estavel (sem mudanca de tendencia relevante na janela)"
        resultado["baseline"] = round(a["media"], 2)
        resultado["amostras"] = amostras
        return resultado

    tc = tempos[k]
    resultado.update({
        "tendencia": "mudanca detectada",
        "baseline_antes_da_mudanca": round(a["a"], 2),
        "mudanca_de_tendencia": min_para_hhmm(tc),
        "margem_de_erro_min": tolerancia,
        "inclinacao_apos_mudanca_por_hora": round(a["b"] * 60, 2),
        "variacao_pct_vs_baseline": round((valores[-1] - a["a"]) / a["a"] * 100, 1) if a["a"] else None,
        "eventos_proximos_da_mudanca": _eventos_proximos(tc, passo),
    })
    antes = [_resumo_mudanca(m, x) for m, x in av.items()
             if m != metrica and x["houve"] and x["tempos"][x["k"]] < tc - tolerancia]
    if antes:   # protecao contra confundir sintoma com causa
        resultado["atencao_outras_metricas_deste_servico_mudaram_ANTES"] = antes
    resultado["amostras"] = amostras
    return resultado

### Teste da nova tool SEM usar a IA (Print: mostra que a ferramenta funciona sozinha)

In [9]:
carregar_cenario("C")
print("=== Varredura do catalog-api (sem informar a metrica) ===")
print(json.dumps(analisar_serie_temporal("catalog-api"), indent=2, ensure_ascii=False))
print("\n=== Apenas a LATENCIA do catalog-api: a tool avisa que outra metrica mudou antes ===")
r = analisar_serie_temporal("catalog-api", "latencia_p95_ms")
r.pop("amostras")
print(json.dumps(r, indent=2, ensure_ascii=False))

=== Varredura do catalog-api (sem informar a metrica) ===
{
  "servico": "catalog-api",
  "janela": "11:00-16:15",
  "resolucao_min": 5,
  "modo": "varredura de todas as metricas do servico",
  "ordem_de_mudanca": [
    {
      "metrica": "memoria_pct",
      "mudanca_de_tendencia": "12:05",
      "baseline_antes_da_mudanca": 54.81,
      "valor_atual": 91.53,
      "variacao_pct_vs_baseline": 67.0,
      "inclinacao_apos_mudanca_por_hora": 8.77
    },
    {
      "metrica": "latencia_p95_ms",
      "mudanca_de_tendencia": "15:30",
      "baseline_antes_da_mudanca": 89.66,
      "valor_atual": 617.62,
      "variacao_pct_vs_baseline": 588.8,
      "inclinacao_apos_mudanca_por_hora": 716.54
    },
    {
      "metrica": "taxa_erro_pct",
      "mudanca_de_tendencia": "15:30",
      "baseline_antes_da_mudanca": 0.2,
      "valor_atual": 2.42,
      "variacao_pct_vs_baseline": 1110.0,
      "inclinacao_apos_mudanca_por_hora": 3.01
    },
    {
      "metrica": "cpu_pct",
      "mudanca_de_

## Declaração das tools no formato Gemini

In [10]:
def decl_servico(nome, descricao):
    return {
        "name": nome,
        "description": descricao,
        "parameters": {
            "type": "OBJECT",
            "properties": {"servico": {"type": "STRING", "enum": SERVICOS, "description": "Nome do servico"}},
            "required": ["servico"],
        },
    }


DECLARACOES = [
    decl_servico("consultar_metricas", "Consulta CPU, memoria, latencia, erros e numero de instancias de um servico (valores atuais)."),
    decl_servico("buscar_logs", "Busca as linhas recentes de log de um servico. Use para investigar sintomas e erros."),
    decl_servico("consultar_deploys", "Lista deploys recentes e mudancas conhecidas de um servico."),
    decl_servico("consultar_configuracao", "Consulta a configuracao operacional atual de um servico."),
    decl_servico("consultar_custos", "Compara o custo atual por hora de um servico com seu baseline."),
    {
        "name": "consultar_eventos_recentes",
        "description": "Lista eventos cloud dos ultimos 90 minutos: deploys, autoscaling e alertas. Eventos mais antigos nao aparecem.",
    },
    {
        "name": "analisar_serie_temporal",
        "description": ("Analisa a evolucao das metricas de um servico ao longo do tempo. Sem informar a metrica, varre as 4 metricas "
                        "e mostra a ORDEM em que mudaram (a que muda primeiro costuma ser a origem; as outras, consequencia), com os "
                        "deploys/eventos proximos da primeira mudanca. Informando a metrica, mostra quando ela mudou e avisa se outra "
                        "metrica do mesmo servico mudou ANTES. Recomendado: comece sem informar a metrica."),
        "parameters": {
            "type": "OBJECT",
            "properties": {
                "servico": {"type": "STRING", "enum": SERVICOS, "description": "Nome do servico"},
                "metrica": {"type": "STRING", "enum": METRICAS, "description": "Opcional. Se omitida, varre todas as metricas do servico"},
            },
            "required": ["servico"],
        },
    },
    {
        "name": "solicitar_rollback",
        "description": "Solicita rollback de um deploy suspeito. Apenas abre pedido; exige aprovacao humana para executar.",
        "parameters": {
            "type": "OBJECT",
            "properties": {
                "deploy_id": {"type": "STRING", "description": "ID exato de um deploy consultado"},
                "justificativa": {"type": "STRING", "description": "Evidencia objetiva que justifica o rollback"},
            },
            "required": ["deploy_id", "justificativa"],
        },
    },
]

FUNCOES_PERMITIDAS = {
    "consultar_metricas": consultar_metricas,
    "buscar_logs": buscar_logs,
    "consultar_deploys": consultar_deploys,
    "consultar_configuracao": consultar_configuracao,
    "consultar_custos": consultar_custos,
    "consultar_eventos_recentes": consultar_eventos_recentes,
    "analisar_serie_temporal": analisar_serie_temporal,
    "solicitar_rollback": solicitar_rollback,
}


def executar_ferramenta(nome, argumentos):
    if nome not in FUNCOES_PERMITIDAS:
        return {"erro": f"Ferramenta nao permitida: {nome}"}
    try:
        return FUNCOES_PERMITIDAS[nome](**argumentos)
    except Exception as exc:
        return {"erro": type(exc).__name__, "detalhe": str(exc)}


print(f"{len(DECLARACOES)} ferramentas declaradas.")

8 ferramentas declaradas.


## Instruções do agente (melhoradas)

Além da regra temporal, acrescentei três regras que vieram de **falhas observadas** na execução do cenário original:
(1) o agente pediu rollback sem ler o conteúdo do deploy; (2) escreveu o ID como `deploy-442` em vez de `dep-442`;
(3) afirmou "memory exhaustion" quando a memória estava em 84%.

In [11]:
INSTRUCOES_BASE = '''
Voce e o agente legista de incidentes Cloud da Loja Nebulosa.
Seu objetivo e investigar a causa raiz usando apenas evidencias retornadas pelas ferramentas.

Regras:
- Nao trate proximidade temporal como prova de causalidade.
- Investigue primeiro; nao chute valores ou eventos.
- Faca consultas curtas e seletivas.
- Diferencie sintoma, causa raiz e impacto financeiro.
- Antes de solicitar rollback, confira o conteudo do deploy com consultar_deploys.
- Cite IDs de deploy exatamente como retornados pelas ferramentas e nao afirme nada que as ferramentas nao mostraram.
- Quando a evidencia apontar um deploy como causa, chame solicitar_rollback antes de encerrar (o pedido apenas abre uma aprovacao humana e e seguro), com justificativa objetiva.
- Nao repita chamadas identicas.
- Nunca declare confianca de 100% nem trate a causa como confirmada alem do que as ferramentas mostraram; cite o que ainda nao foi verificado.
- Nunca diga que uma acao foi executada quando a ferramenta disser que aguarda aprovacao.
- Encerre com: diagnostico, evidencias, impacto, acao recomendada e nivel de confianca.
- Seja conciso.
'''.strip()

INSTRUCAO_TEMPORAL = '''
- Quando o problema for gradual, aparecer com atraso ou houver varios deploys candidatos, use analisar_serie_temporal para descobrir
  QUANDO cada metrica mudou. Um evento so pode ser causa se ocorreu ANTES do inicio do sintoma. Lembre que consultar_eventos_recentes
  mostra apenas os ultimos 90 minutos.
- Analise tambem o servico do qual o servico sintomatico depende (veja os logs de timeout), e nao apenas o servico do alerta.
- Nao atribua a causa ao deploy mais proximo do SINTOMA (por exemplo, a latencia). Antes, analise o servico suspeito sem informar a
  metrica e veja qual metrica mudou PRIMEIRO; se uma metrica de recurso (memoria, cpu) ja vinha mudando antes, a causa esta no inicio dessa mudanca.
- REGRA DE DECISAO: o deploy candidato a rollback e o que coincide com o INICIO da primeira mudanca. Um deploy POSTERIOR so pode ser tratado como agravante se a ferramenta mostrar alterou_a_inclinacao=true; sem essa evidencia, nao solicite rollback dele (no maximo, cite-o como acao secundaria no relatorio).
'''.strip()


def montar_instrucoes(usar_tool_temporal):
    return INSTRUCOES_BASE + ("\n" + INSTRUCAO_TEMPORAL if usar_tool_temporal else "")

## Chamada à API e laço do agente

Novidades em relação à v1.1: o agente registra todas as chamadas (`chamadas`) para a avaliação, pode rodar **sem** a tool temporal
(`usar_tool_temporal=False`, para comparar) e pode rodar em modo silencioso (`verboso=False`).

In [12]:
LOG_RETRY = True
MAX_TENTATIVAS_API = 6
TIMEOUT_API_S = 120
CONTADORES_API = {"retentativas": 0}
RELATORIO_MIN_CHARS = 350
PEDIDO_RELATORIO_COMPLETO = (
    "Seu relatorio final esta incompleto. Reescreva-o no formato pedido: diagnostico, evidencias (citando IDs e horarios "
    "exatamente como as ferramentas retornaram), impacto, acao recomendada e nivel de confianca."
)


class ErroAPI(RuntimeError):
    def __init__(self, codigo, detalhe):
        super().__init__(f"Erro {codigo} da API Gemini: {detalhe}")
        self.codigo = codigo


def chamar_gemini(corpo, tentativas=MAX_TENTATIVAS_API):
    url = f"{BASE_URL}/models/{MODEL}:generateContent"
    for tentativa in range(1, tentativas + 1):
        req = urllib.request.Request(
            url,
            data=json.dumps(corpo).encode("utf-8"),
            method="POST",
            headers={"Content-Type": "application/json", "X-goog-api-key": os.environ["GEMINI_API_KEY"]},
        )
        try:
            with urllib.request.urlopen(req, timeout=TIMEOUT_API_S) as resposta:
                return json.loads(resposta.read())
        except urllib.error.HTTPError as erro:
            detalhe = erro.read().decode("utf-8", errors="replace")[:500]
            config = corpo.get("generationConfig", {})
            if erro.code == 400 and "thinking" in detalhe.lower() and "thinkingConfig" in config:
                del config["thinkingConfig"]
                continue
            if erro.code in (429, 500, 503) and tentativa < tentativas:
                espera = min(15 * tentativa, 60)
                CONTADORES_API["retentativas"] += 1
                if LOG_RETRY:
                    print(f"  (API respondeu {erro.code}; aguardando {espera}s para tentar de novo...)")
                time.sleep(espera)
                continue
            raise ErroAPI(erro.code, detalhe) from None
        except (urllib.error.URLError, OSError, ValueError) as erro:   # timeout de leitura, queda de conexao, corpo truncado
            if tentativa < tentativas:
                espera = min(15 * tentativa, 60)
                CONTADORES_API["retentativas"] += 1
                if LOG_RETRY:
                    print(f"  (falha de rede/timeout: {type(erro).__name__}; aguardando {espera}s para tentar de novo...)")
                time.sleep(espera)
                continue
            raise ErroAPI("rede", type(erro).__name__) from None


def estimar_custo(input_tokens, output_tokens):
    return input_tokens * PRECO_INPUT_1M / 1_000_000 + output_tokens * PRECO_OUTPUT_1M / 1_000_000


def investigar_incidente(incidente=None, usar_tool_temporal=True, verboso=True):
    def falar(*args):
        if verboso:
            print(*args)

    incidente = incidente or ambiente["incidente"]
    declaracoes = [d for d in DECLARACOES if usar_tool_temporal or d["name"] != "analisar_serie_temporal"]
    historico = [{"role": "user", "parts": [{"text": incidente}]}]
    total_input = total_output = 0
    chamadas = []
    ja_feitas = set()
    repetidas = 0
    pediu_relatorio_completo = False
    respostas_vazias = 0

    def resumo(texto_final, interrompido, rodada, motivo, finish=None):
        return {
            "resposta": texto_final,
            "interrompido": interrompido,
            "motivo_fim": motivo,
            "finish_reason": finish,
            "rodadas": rodada,
            "input_tokens": total_input,
            "output_tokens": total_output,
            "custo_equivalente_usd": estimar_custo(total_input, total_output),
            "ferramentas": [nome for nome, _ in chamadas],
            "chamadas": chamadas,
            "chamadas_repetidas": repetidas,
        }

    for rodada in range(1, MAX_RODADAS + 1):
        ultima = rodada == MAX_RODADAS
        if ultima:
            falar("(ultima rodada: ferramentas desativadas; o agente precisa escrever o relatorio final)")
        corpo = {
            "systemInstruction": {"parts": [{"text": montar_instrucoes(usar_tool_temporal)}]},
            "contents": historico,
            "tools": [{"functionDeclarations": declaracoes}],
            "toolConfig": {"functionCallingConfig": {"mode": "NONE" if ultima else "AUTO"}},
            "generationConfig": {"maxOutputTokens": MAX_OUTPUT_TOKENS,
                                 "thinkingConfig": {"thinkingLevel": THINKING_LEVEL}},
        }
        resposta = chamar_gemini(corpo)
        uso = resposta.get("usageMetadata", {})
        total_input += uso.get("promptTokenCount", 0)
        total_output += uso.get("candidatesTokenCount", 0) + uso.get("thoughtsTokenCount", 0)
        custo = estimar_custo(total_input, total_output)
        falar(f"\n--- Rodada {rodada} | custo equivalente acumulado: US$ {custo:.6f} (real: US$ 0, plano gratuito) ---")

        candidatos = resposta.get("candidates") or []
        if not candidatos:
            falar("A API nao devolveu candidatos:", json.dumps(resposta.get("promptFeedback", {}), ensure_ascii=False))
            return resumo(None, True, rodada, "sem_candidatos")

        candidato = candidatos[0]
        finish = candidato.get("finishReason")
        conteudo = candidato.get("content", {"role": "model", "parts": []})
        partes = conteudo.get("parts", [])

        if not partes:   # resposta vazia: nao entra no historico (a API rejeitaria) e tentamos de novo
            respostas_vazias += 1
            falar(f"Resposta vazia do modelo (finishReason={finish}); tentando de novo.")
            if respostas_vazias > 2:
                return resumo(None, True, rodada, f"resposta_vazia:{finish}", finish)
            continue

        historico.append(conteudo)  # preserva a thoughtSignature
        pedidos = [p["functionCall"] for p in partes if "functionCall" in p]

        if not pedidos:
            texto = "".join(p.get("text", "") for p in partes if not p.get("thought"))
            if len(texto.strip()) < RELATORIO_MIN_CHARS and not pediu_relatorio_completo and rodada < MAX_RODADAS:
                pediu_relatorio_completo = True
                falar(f"Relatorio curto demais ({len(texto.strip())} caracteres); pedindo a versao completa...")
                historico.append({"role": "user", "parts": [{"text": PEDIDO_RELATORIO_COMPLETO}]})
                continue
            if finish == "MAX_TOKENS":
                falar("AVISO: resposta cortada por MAX_OUTPUT_TOKENS; aumente o limite.")
            falar("\nRELATORIO FINAL\n")
            falar(texto)
            return resumo(texto, False, rodada, "relatorio", finish)

        retornos = []
        for pedido in pedidos:
            nome, argumentos = pedido["name"], pedido.get("args", {})
            chave = (nome, json.dumps(argumentos, sort_keys=True, ensure_ascii=False))
            falar(f"Agente pediu: {nome}({argumentos})")
            if chave in ja_feitas:   # economia: nao reexecuta nem reenvia um resultado que o modelo ja tem
                repetidas += 1
                resultado = {"aviso": "Chamada repetida: este resultado ja foi fornecido antes nesta investigacao. "
                                      "Use uma evidencia nova ou conclua."}
            else:
                ja_feitas.add(chave)
                resultado = executar_ferramenta(nome, argumentos)
            texto_obs = json.dumps(resultado, ensure_ascii=False)
            falar("Observacao:", texto_obs if len(texto_obs) < 700 else texto_obs[:700] + " ...[cortado na tela]")
            chamadas.append((nome, argumentos))
            retorno = {"name": nome, "response": {"result": resultado}}
            if "id" in pedido:
                retorno["id"] = pedido["id"]
            retornos.append({"functionResponse": retorno})
        restantes = MAX_RODADAS - rodada
        if restantes <= 2:   # aviso de orcamento: evita estourar as rodadas sem relatorio
            retornos[-1]["functionResponse"]["response"]["aviso_orcamento"] = (
                f"Restam {restantes} rodadas. Se ja identificou a causa, chame solicitar_rollback agora; "
                "a ultima rodada sera reservada para o relatorio final.")
        historico.append({"role": "user", "parts": retornos})

        if custo >= LIMITE_ESTIMADO_USD:
            falar(f"\nLimite didatico de US$ {LIMITE_ESTIMADO_USD:.2f} (equivalente) atingido.")
            return resumo(None, True, rodada, "limite_custo")

    falar("\nNumero maximo de rodadas atingido. O agente foi interrompido pelo software.")
    return resumo(None, True, MAX_RODADAS, "max_rodadas")

## Execução 1: cenário B (o problema novo), com a tool temporal

In [13]:
carregar_cenario("B")
resultado_B = investigar_incidente()


--- Rodada 1 | custo equivalente acumulado: US$ 0.000493 (real: US$ 0, plano gratuito) ---
Agente pediu: consultar_metricas({'servico': 'checkout-api'})
Observacao: {"servico": "checkout-api", "cpu_pct": 47.98, "memoria_pct": 58.88, "latencia_p95_ms": 1068.99, "taxa_erro_pct": 4.86, "instancias": 8}
Agente pediu: analisar_serie_temporal({'servico': 'checkout-api'})
Observacao: {"servico": "checkout-api", "janela": "11:00-16:15", "resolucao_min": 5, "modo": "varredura de todas as metricas do servico", "ordem_de_mudanca": [{"metrica": "latencia_p95_ms", "mudanca_de_tendencia": "15:40", "baseline_antes_da_mudanca": 308.28, "valor_atual": 1068.99, "variacao_pct_vs_baseline": 246.8, "inclinacao_apos_mudanca_por_hora": 1365.08}, {"metrica": "taxa_erro_pct", "mudanca_de_tendencia": "15:40", "baseline_antes_da_mudanca": 0.59, "valor_atual": 4.86, "variacao_pct_vs_baseline": 720.7, "inclinacao_apos_mudanca_por_hora": 7.51}], "metricas_estaveis": ["cpu_pct", "memoria_pct"], "primeira_metrica_a_

## Execução 2: cenário A (o original), para provar que a tool nova não quebrou o que já funcionava

In [14]:
carregar_cenario("A")
resultado_A = investigar_incidente()


--- Rodada 1 | custo equivalente acumulado: US$ 0.000471 (real: US$ 0, plano gratuito) ---
Agente pediu: consultar_metricas({'servico': 'api-gateway'})
Observacao: {"servico": "api-gateway", "cpu_pct": 38, "memoria_pct": 61, "latencia_p95_ms": 6200, "taxa_erro_pct": 18.4, "instancias": 3}
Agente pediu: consultar_custos({'servico': 'api-gateway'})
Observacao: {"servico": "api-gateway", "baseline_usd_h": 0.22, "atual_usd_h": 0.24, "multiplicador": 1.1}

--- Rodada 2 | custo equivalente acumulado: US$ 0.000942 (real: US$ 0, plano gratuito) ---
Agente pediu: analisar_serie_temporal({'servico': 'api-gateway'})
Observacao: {"servico": "api-gateway", "janela": "13:00-14:08", "resolucao_min": 1, "modo": "varredura de todas as metricas do servico", "ordem_de_mudanca": [{"metrica": "latencia_p95_ms", "mudanca_de_tendencia": "14:02", "baseline_antes_da_mudanca": 139.51, "valor_atual": 6200, "variacao_pct_vs_baseline": 4344.3, "inclinacao_apos_mudanca_por_hora": 76285.75}, {"metrica": "taxa_erro_

## Avaliação automática

Cada caso é executado `REPETICOES` vezes. O critério de acerto é objetivo e verificado pelo código, não por opinião:
- o agente **pediu rollback do deploy correto** (A: `dep-442`; B: `dep-451`);
- e **não pediu rollback do deploy errado** (B: `dep-455`, a pista falsa);
- e terminou com um relatório (não foi interrompido).

Também registramos se usou a tool temporal, quantas rodadas gastou, tokens e custo equivalente.
Os resultados variam de uma execução para outra (o modelo não é 100% determinístico), por isso repetimos.

In [15]:
REPETICOES = 3
ARQUIVO_RESULTADOS = "resultados_avaliacao_v3_2.json"
ARQUIVO_PARCIAL = "resultados_parciais_v3_2.json"   # salvo apos CADA execucao (nada se perde se der erro)

CASOS = [
    # nome, cenario, usar_tool_temporal, rollback esperado, rollbacks proibidos
    ("A original, com tool temporal", "A", True, "dep-442", []),
    ("C confuso, SEM tool temporal", "C", False, "dep-451", ["dep-453", "dep-455"]),
    ("C confuso, COM tool temporal", "C", True, "dep-451", ["dep-453", "dep-455"]),
    ("B facil, SEM tool temporal", "B", False, "dep-451", ["dep-455"]),
    ("B facil, COM tool temporal", "B", True, "dep-451", ["dep-455"]),
]
CASOS_PADRAO = [c[0] for c in CASOS[:3]]   # os dois de B ja foram medidos na 1a rodada de testes


def _vazio(motivo, infra):
    return {"acertou": False, "infra": infra, "motivo": motivo, "rollbacks": [], "cita_causa": False,
            "usou_temporal": False, "rodadas": 0, "tokens": 0, "custo": 0.0, "ferramentas": [],
            "chamadas_repetidas": 0, "relatorio": ""}


def rodar_um_caso(cenario, usar_tool, esperado, proibidos, tentativas_infra=3):
    carregar_cenario(cenario)
    for tentativa in range(1, tentativas_infra + 1):
        try:
            r = investigar_incidente(usar_tool_temporal=usar_tool, verboso=False)
            break
        except ErroAPI as exc:
            if tentativa == tentativas_infra:
                return _vazio(f"erro_api_{exc.codigo}", True)
            time.sleep(30)   # servidor do Google sobrecarregado: espera e repete a execucao inteira
        except Exception as exc:   # qualquer outro erro vira um resultado registrado, nao derruba a avaliacao
            return _vazio(f"erro_inesperado_{type(exc).__name__}", False)
    rollbacks = [args.get("deploy_id") for nome, args in r["chamadas"] if nome == "solicitar_rollback"]
    acertou = (not r["interrompido"]) and esperado in rollbacks and not any(p in rollbacks for p in proibidos)
    if acertou:
        motivo = "ok"
    elif r["interrompido"]:
        motivo = r["motivo_fim"]
    elif not rollbacks:
        motivo = "terminou_sem_pedir_rollback"
    else:
        motivo = "rollback_errado"
    texto = r["resposta"] or ""
    return {
        "acertou": acertou, "infra": False, "motivo": motivo, "rollbacks": rollbacks,
        "cita_causa": esperado in texto,   # o relatorio final cita o deploy correto?
        "usou_temporal": "analisar_serie_temporal" in r["ferramentas"],
        "rodadas": r["rodadas"], "tokens": r["input_tokens"] + r["output_tokens"],
        "custo": r["custo_equivalente_usd"], "ferramentas": r["ferramentas"],
        "chamadas_repetidas": r["chamadas_repetidas"], "finish_reason": r["finish_reason"],
        "tamanho_relatorio": len(texto), "relatorio": texto,
    }


def _carregar_parciais():
    try:
        with open(ARQUIVO_PARCIAL, encoding="utf-8") as f:
            return json.load(f)
    except (OSError, ValueError):
        return {}


def avaliar_tudo(repeticoes=REPETICOES, casos=None, retomar=True):
    global LOG_RETRY
    LOG_RETRY = False            # evita encher a tela; as retentativas sao contadas
    nomes = casos or CASOS_PADRAO
    selecionados = [c for c in CASOS if c[0] in nomes]
    tabela, detalhes = [], []
    parciais = _carregar_parciais() if retomar else {}
    try:
        for nome, cenario, usar_tool, esperado, proibidos in selecionados:
            CONTADORES_API["retentativas"] = 0
            # reaproveita execucoes ja concluidas (descarta as que falharam por infraestrutura)
            execucoes = [x for x in parciais.get(nome, []) if not x["infra"]][:repeticoes]
            if execucoes:
                print(f"{nome}: reaproveitando {len(execucoes)} execucao(oes) ja salva(s).")
            while len(execucoes) < repeticoes:
                print(f"Executando: {nome} (tentativa {len(execucoes) + 1}/{repeticoes})...")
                execucoes.append(rodar_um_caso(cenario, usar_tool, esperado, proibidos))
                parciais[nome] = execucoes
                with open(ARQUIVO_PARCIAL, "w", encoding="utf-8") as f:
                    json.dump(parciais, f, ensure_ascii=False, indent=1)
                time.sleep(3)
            validas = [x for x in execucoes if not x["infra"]]
            media = lambda campo: sum(x[campo] for x in validas) / max(1, len(validas))
            tabela.append({
                "caso": nome,
                "acertos_rollback": f"{sum(x['acertou'] for x in execucoes)}/{repeticoes}",
                "cita_causa_no_relatorio": f"{sum(x['cita_causa'] for x in execucoes)}/{repeticoes}",
                "execucoes_validas": f"{len(validas)}/{repeticoes}",
                "usou_temporal": f"{sum(x['usou_temporal'] for x in execucoes)}/{repeticoes}",
                "rodadas_media": round(media("rodadas"), 1),
                "tokens_media": round(media("tokens")),
                "custo_eq_media_usd": round(media("custo"), 5),
                "chamadas_repetidas_media": round(media("chamadas_repetidas"), 1),
                "retentativas_api": CONTADORES_API["retentativas"],
            })
            detalhes.append({"caso": nome, "execucoes": execucoes})
    finally:
        LOG_RETRY = True

    print("\n" + "=" * 118)
    print(f"{'Caso':32} {'Rollback':9} {'Menciona':11} {'Usou temp.':11} {'Rodadas':8} {'Tokens':8} {'Custo eq.US$':13} {'Repetidas':10} {'Retent.API':10}")
    for t in tabela:
        print(f"{t['caso']:32} {t['acertos_rollback']:9} {t['cita_causa_no_relatorio']:11} {t['usou_temporal']:11} "
              f"{t['rodadas_media']:<8} {t['tokens_media']:<8} {t['custo_eq_media_usd']:<13} {t['chamadas_repetidas_media']:<10} {t['retentativas_api']:<10}")
    print("=" * 118)
    for d in detalhes:
        print(d["caso"])
        for i, x in enumerate(d["execucoes"], 1):
            print(f"   #{i}: {x['motivo']:28} rollbacks={x['rollbacks']} cita_causa={x['cita_causa']} rodadas={x['rodadas']} tools={len(x['ferramentas'])}")
    with open(ARQUIVO_RESULTADOS, "w", encoding="utf-8") as f:
        json.dump({"modelo": MODEL, "tabela": tabela, "detalhes": detalhes}, f, ensure_ascii=False, indent=1)
    print(f"\nDetalhes completos salvos em {ARQUIVO_RESULTADOS}")
    return tabela

In [16]:
tabela = avaliar_tudo()

Executando: A original, com tool temporal (tentativa 1/3)...
Executando: A original, com tool temporal (tentativa 2/3)...
Executando: A original, com tool temporal (tentativa 3/3)...
Executando: C confuso, SEM tool temporal (tentativa 1/3)...
Executando: C confuso, SEM tool temporal (tentativa 2/3)...
Executando: C confuso, SEM tool temporal (tentativa 3/3)...
Executando: C confuso, COM tool temporal (tentativa 1/3)...
Executando: C confuso, COM tool temporal (tentativa 2/3)...
Executando: C confuso, COM tool temporal (tentativa 3/3)...

Caso                             Rollback  Menciona    Usou temp.  Rodadas  Tokens   Custo eq.US$  Repetidas  Retent.API
A original, com tool temporal    3/3       3/3         3/3         5.7      17554    0.00529       0.0        6         
C confuso, SEM tool temporal     0/3       0/3         0/3         5.3      11345    0.00375       0.0        11        
C confuso, COM tool temporal     0/3       0/3         3/3         6.0      18494    0.00504 

In [17]:
MODEL = "gemini-3.5-flash"
res = [rodar_um_caso("C", True, "dep-451", ["dep-453", "dep-455"]) for _ in range(2)]
for x in res:
    print(x["motivo"], x["rollbacks"], "| rodadas:", x["rodadas"], "| tokens:", x["tokens"])
MODEL = "gemini-3.1-flash-lite"

  (API respondeu 503; aguardando 15s para tentar de novo...)
  (API respondeu 503; aguardando 15s para tentar de novo...)
  (API respondeu 503; aguardando 15s para tentar de novo...)
ok ['dep-451'] | rodadas: 8 | tokens: 29432
max_rodadas ['dep-451'] | rodadas: 8 | tokens: 24469


## Roteiro do professor: situação

1. Tool de análise temporal: **feito** (`analisar_serie_temporal`)
   1.1. Mockup de métricas com problema novo que exige investigação temporal: **feito** (cenário B)
2. Testar e avaliar: **feito** (execuções 1 e 2 e tabela de avaliação)
3. Portar para nova LLM: **feito** (OpenAI -> Gemini)